In [1]:
import pandas as pd
import os
from collections import defaultdict

In [2]:
def generate_supplementary_table(
    dataset_name,
    results_dir,
    deg_level=20,
    scenario=None,
    output_path=None,
):
    """
    generate one Supp Excel file for a dataset
    values are averaged across seeds
    """

    # paths
    base_path = os.path.join(results_dir, dataset_name)
    if scenario is not None:
        base_path = os.path.join(base_path, scenario)

    
    methods = sorted([
        d for d in os.listdir(base_path)
        if os.path.isdir(os.path.join(base_path, d))
        and not d.startswith('.')
    ])

    # read all csvs
    #  {method: {ood_label: {metric_name: [values across seeds]}}}
    raw = {}

    for method in methods:
        method_path = os.path.join(base_path, method)
        csv_files = [f for f in os.listdir(method_path) if f.endswith('.csv')]
        method_raw = defaultdict(lambda: defaultdict(list))

        for csv_file in csv_files:
            stem = csv_file.replace('.csv', '')

            # parse: CD14 Mono_3: ood="CD14 Mono", seed=3
            # last underscore separates seed from OOD label
            parts = stem.rsplit('_', 1)
            if len(parts) == 2 and parts[1].isdigit():
                ood_label = parts[0]
            else:
                ood_label = stem  # No seed (Norman)

            df = pd.read_csv(os.path.join(method_path, csv_file))
            deg_col = str(deg_level)

            for _, row in df.iterrows():
                metric_name = row['Metric']
                value = row.get(deg_col, None)
                if value is not None and pd.notna(value):
                    method_raw[ood_label][metric_name].append(float(value))

        raw[method] = method_raw

    # avg across seeds
    # {method: {ood_label: {metric_name: mean_value}}}
    averaged = {}
    for method, ood_dict in raw.items():
        averaged[method] = {}
        for ood_label, metric_dict in ood_dict.items():
            averaged[method][ood_label] = {
                metric: sum(vals) / len(vals)
                for metric, vals in metric_dict.items()
            }

    # collect all ood labels and metrics
    all_ood_labels = sorted(set(
        ood for method_data in averaged.values()
        for ood in method_data.keys()
    ))
    all_metrics = sorted(set(
        metric for method_data in averaged.values()
        for ood_data in method_data.values()
        for metric in ood_data.keys()
    ))

    #  to excel (1 excel sheet per metric)
    if output_path is None:
        output_path = f'{dataset_name}_supplementary.xlsx'

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        for metric in all_metrics:
            rows = []
            for method in methods:
                row = {'Method': method}
                values_for_avg = []

                for ood_label in all_ood_labels:
                    val = averaged[method].get(ood_label, {}).get(metric, None)
                    if val is not None:
                        row[ood_label] = round(val, 2)
                        values_for_avg.append(val)
                    else:
                        row[ood_label] = ''

                if values_for_avg:
                    row['Average'] = round(
                        sum(values_for_avg) / len(values_for_avg), 2
                    )
                else:
                    row['Average'] = ''

                rows.append(row)

            sheet_df = pd.DataFrame(rows)
            # sheet names limited to 31 char
            sheet_name = metric[:31]
            sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f'Saved: {output_path}')
    print(f'Methods: {methods}')
    print(f'OOD conditions: {all_ood_labels}')
    print(f'Metrics (sheets): {all_metrics}')

In [3]:
results_dir = '../Benchmarks/y_pred_results/results'

In [4]:
# Kang
generate_supplementary_table('Kang', results_dir, deg_level=20,
    output_path='Kang_supplementary.xlsx')

Saved: Kang_supplementary.xlsx
Methods: ['BIOLORD', 'CPA', 'Ctrl Mean', 'Pert Mean', 'SCDISENTANGLE', 'SCDISINFACT', 'SCGEN']
OOD conditions: ['B', 'CD14 Mono', 'CD16 Mono', 'CD4 T', 'CD8 T', 'DC', 'NK', 'T']
Metrics (sheets): ['Normalized RMSE Mean', 'Pearson Mean', 'Pearson Mean Delta', 'R2 Mean', 'R2 Mean Delta', 'RMSE Mean', 'Single-cell Identity preservation', 'Wasserstein']


In [5]:
# Liver
generate_supplementary_table('Liver', results_dir, deg_level=20,
    output_path='Liver_supplementary.xlsx')

Saved: Liver_supplementary.xlsx
Methods: ['BIOLORD', 'CPA', 'Ctrl Mean', 'Pert Mean', 'SCDISENTANGLE', 'SCDISINFACT', 'SCGEN']
OOD conditions: ['Pericentral', 'Periportal']
Metrics (sheets): ['Normalized RMSE Mean', 'Pearson Mean', 'Pearson Mean Delta', 'R2 Mean', 'R2 Mean Delta', 'RMSE Mean', 'Single-cell Identity preservation', 'Wasserstein']


In [6]:
# Myocarditis CD8
generate_supplementary_table('Myocarditis_CD8', results_dir, deg_level=20,
    output_path='Myocarditis_CD8_supplementary.xlsx')

Saved: Myocarditis_CD8_supplementary.xlsx
Methods: ['Ctrl Mean', 'Pert Mean', 'SCDISENTANGLE']
OOD conditions: ['SIC_153', 'SIC_164', 'SIC_171', 'SIC_175', 'SIC_177', 'SIC_199', 'SIC_217', 'SIC_232', 'SIC_258', 'SIC_264', 'SIC_48']
Metrics (sheets): ['Normalized RMSE Mean', 'Pearson Mean', 'Pearson Mean Delta', 'Perturbed Reference Pearson Delta', 'Perturbed Reference R2 Delta', 'R2 Mean', 'R2 Mean Delta', 'RMSE Mean', 'RMSE Mean Delta', 'Relative Wasserstein Distance', 'Single-cell Identity preservation', 'Wasserstein']


In [7]:
# Myocarditis full
generate_supplementary_table('Myocarditis', results_dir, deg_level=20,
    output_path='Myocarditis_full_supplementary.xlsx')

Saved: Myocarditis_full_supplementary.xlsx
Methods: ['Ctrl Mean', 'Pert Mean', 'SCDISENTANGLE']
OOD conditions: ['SIC_153', 'SIC_164', 'SIC_171', 'SIC_175', 'SIC_177', 'SIC_199', 'SIC_217', 'SIC_232', 'SIC_258', 'SIC_264', 'SIC_48']
Metrics (sheets): ['Normalized RMSE Mean', 'Pearson Mean', 'Pearson Mean Delta', 'Perturbed Reference Pearson Delta', 'Perturbed Reference R2 Delta', 'R2 Mean', 'R2 Mean Delta', 'RMSE Mean', 'RMSE Mean Delta', 'Relative Wasserstein Distance', 'Single-cell Identity preservation', 'Wasserstein']


In [8]:
# Norman combinatorial standard DEGs
generate_supplementary_table('Norman', results_dir, deg_level=20,
    scenario='combinatorially_seen',
    output_path='Norman_combinatorially_seen_standard_supplementary.xlsx')

Saved: Norman_combinatorially_seen_standard_supplementary.xlsx
Methods: ['CPA', 'GEARS', 'Match Mean', 'Pert Mean', 'SCDISENTANGLE']
OOD conditions: ['AHR+FEV', 'AHR+KLF1', 'BCL2L11+BAK1', 'BCL2L11+TGFBR2', 'BPGM+SAMD1', 'BPGM+ZBTB1', 'CBL+CNN1', 'CBL+PTPN12', 'CBL+PTPN9', 'CBL+TGFBR2', 'CBL+UBASH3A', 'CBL+UBASH3B', 'CDKN1B+CDKN1A', 'CDKN1C+CDKN1A', 'CDKN1C+CDKN1B', 'CEBPB+CEBPA', 'CEBPB+MAPK1', 'CEBPB+OSR2', 'CEBPB+PTPN12', 'CEBPE+CEBPA', 'CEBPE+CEBPB', 'CEBPE+CNN1', 'CEBPE+KLF1', 'CEBPE+PTPN12', 'CEBPE+RUNX1T1', 'CEBPE+SPI1', 'CNN1+MAPK1', 'CNN1+UBASH3A', 'DUSP9+ETS2', 'DUSP9+IGDCC3', 'DUSP9+KLF1', 'DUSP9+MAPK1', 'DUSP9+PRTG', 'DUSP9+SNAI1', 'ETS2+CEBPE', 'ETS2+CNN1', 'ETS2+IGDCC3', 'ETS2+IKZF3', 'ETS2+MAP7D1', 'ETS2+MAPK1', 'ETS2+PRTG', 'FEV+CBFA2T3', 'FEV+ISL2', 'FEV+MAP7D1', 'FOSB+CEBPB', 'FOSB+CEBPE', 'FOSB+IKZF3', 'FOSB+OSR2', 'FOSB+PTPN12', 'FOSB+UBASH3B', 'FOXA1+FOXF1', 'FOXA1+FOXL2', 'FOXA1+HOXB9', 'FOXA3+FOXA1', 'FOXA3+FOXF1', 'FOXA3+FOXL2', 'FOXA3+HOXB9', 'FOXF1+FOXL2', 'FO

In [9]:
# Norman combinatorial combo-specific DEGs
generate_supplementary_table('Norman', results_dir, deg_level=20,
    scenario='combinatorially_seen_combo_specific',
    output_path='Norman_combinatorially_seen_combo_specific_supplementary.xlsx')

Saved: Norman_combinatorially_seen_combo_specific_supplementary.xlsx
Methods: ['CPA', 'GEARS', 'Match Mean', 'Pert Mean', 'SCDISENTANGLE']
OOD conditions: ['AHR+FEV', 'AHR+KLF1', 'BCL2L11+BAK1', 'BCL2L11+TGFBR2', 'BPGM+SAMD1', 'BPGM+ZBTB1', 'CBL+CNN1', 'CBL+PTPN12', 'CBL+PTPN9', 'CBL+TGFBR2', 'CBL+UBASH3A', 'CBL+UBASH3B', 'CDKN1B+CDKN1A', 'CDKN1C+CDKN1A', 'CDKN1C+CDKN1B', 'CEBPB+CEBPA', 'CEBPB+MAPK1', 'CEBPB+OSR2', 'CEBPB+PTPN12', 'CEBPE+CEBPA', 'CEBPE+CEBPB', 'CEBPE+CNN1', 'CEBPE+KLF1', 'CEBPE+PTPN12', 'CEBPE+RUNX1T1', 'CEBPE+SPI1', 'CNN1+MAPK1', 'CNN1+UBASH3A', 'DUSP9+ETS2', 'DUSP9+IGDCC3', 'DUSP9+KLF1', 'DUSP9+MAPK1', 'DUSP9+PRTG', 'DUSP9+SNAI1', 'ETS2+CEBPE', 'ETS2+CNN1', 'ETS2+IGDCC3', 'ETS2+IKZF3', 'ETS2+MAP7D1', 'ETS2+MAPK1', 'ETS2+PRTG', 'FEV+CBFA2T3', 'FEV+ISL2', 'FEV+MAP7D1', 'FOSB+CEBPB', 'FOSB+CEBPE', 'FOSB+IKZF3', 'FOSB+OSR2', 'FOSB+PTPN12', 'FOSB+UBASH3B', 'FOXA1+FOXF1', 'FOXA1+FOXL2', 'FOXA1+HOXB9', 'FOXA3+FOXA1', 'FOXA3+FOXF1', 'FOXA3+FOXL2', 'FOXA3+HOXB9', 'FOXF1+FOXL2

In [10]:
# Norman single-only standard DEGs
generate_supplementary_table('Norman', results_dir, deg_level=20,
    scenario='single_only',
    output_path='Norman_single_only_standard_supplementary.xlsx')

Saved: Norman_single_only_standard_supplementary.xlsx
Methods: ['CPA', 'GEARS', 'Match Mean', 'Pert Mean', 'SCDISENTANGLE']
OOD conditions: ['AHR+FEV', 'CBL+CNN1', 'CBL+PTPN12', 'CEBPB+CEBPA', 'CNN1+MAPK1', 'DUSP9+ETS2', 'DUSP9+MAPK1', 'ETS2+CEBPE', 'ETS2+MAPK1']
Metrics (sheets): ['Centroid Accuracy', 'MAE Accuracy', 'Normalized RMSE Mean', 'Pearson Mean', 'Pearson Mean Delta', 'Perturbed Reference Pearson Delta', 'Perturbed Reference R2 Delta', 'R2 Mean', 'R2 Mean Delta', 'RMSE Mean', 'RMSE Mean Delta', 'Relative Wasserstein Distance', 'Single-cell Identity preservation', 'Wasserstein', 'Wasserstein Accuracy']


In [11]:
# Norman single-only combo-specific DEGs
generate_supplementary_table('Norman', results_dir, deg_level=20,
    scenario='single_only_combo_specific',
    output_path='Norman_single_only_combo_specific_supplementary.xlsx')

Saved: Norman_single_only_combo_specific_supplementary.xlsx
Methods: ['CPA', 'GEARS', 'Match Mean', 'Pert Mean', 'SCDISENTANGLE']
OOD conditions: ['AHR+FEV', 'CBL+CNN1', 'CBL+PTPN12', 'CEBPB+CEBPA', 'CNN1+MAPK1', 'DUSP9+ETS2', 'DUSP9+MAPK1', 'ETS2+CEBPE', 'ETS2+MAPK1']
Metrics (sheets): ['Centroid Accuracy', 'MAE Accuracy', 'Normalized RMSE Mean', 'Pearson Mean', 'Pearson Mean Delta', 'Perturbed Reference Pearson Delta', 'Perturbed Reference R2 Delta', 'R2 Mean', 'R2 Mean Delta', 'RMSE Mean', 'RMSE Mean Delta', 'Relative Wasserstein Distance', 'Single-cell Identity preservation', 'Wasserstein', 'Wasserstein Accuracy']
